# LC 338 — Counting Bits
**Difficulty:** Easy | **Category:** Bit Manipulation
**Pattern:** DP + Bit Shift Recurrence

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Shifting i right by 1 gives you i//2,
which you already solved. The last bit (i&1) is either 0 or 1.
So <code>dp[i] = dp[i >> 1] + (i & 1)</code> — build from smaller
answers you already have.
</div>

## Official Problem Statement

Given an integer `n`, return an array `ans` of length `n + 1` such
that for each `i` (0 <= i <= n), `ans[i]` is the number of 1's
in the binary representation of `i`.

**Constraints:**
- `0 <= n <= 10^5`

**Follow-up:**
- Solve in O(n) time and O(n) space
- Use only O(1) extra space beyond the output array

## What This Is Actually Asking

Build a lookup table: for every number from 0 to n,
how many 1-bits does it have?

The result for n=5 is [0,1,1,2,1,2] — index i holds the bit count
of the number i.

The key insight is that each number's bit count relates
to a smaller number's bit count you already computed.

## Walk Through an Example by Hand

n = 5. Build dp[0..5]:

```
i=0: 0000 → dp[0] = 0           (base case)
i=1: 0001 → dp[1>>1] + (1&1)
           = dp[0]   +    1  = 1
i=2: 0010 → dp[2>>1] + (2&1)
           = dp[1]   +    0  = 1
i=3: 0011 → dp[3>>1] + (3&1)
           = dp[1]   +    1  = 2
i=4: 0100 → dp[4>>1] + (4&1)
           = dp[2]   +    0  = 1
i=5: 0101 → dp[5>>1] + (5&1)
           = dp[2]   +    1  = 2
```
Result: [0, 1, 1, 2, 1, 2]

## The Picture

```
Halving i removes its last bit:

  i = 5  →  0 1 0 1   (2 ones)
             ↓
  i>>1 = 2  →  0 0 1 0   (1 one)  ← already in dp
  last bit of 5:  5 & 1 = 1       ← add 1
  dp[5] = dp[2] + 1 = 2  ✓

  i = 6  →  0 1 1 0   (2 ones)
             ↓
  i>>1 = 3  →  0 0 1 1   (2 ones)  ← already in dp
  last bit of 6:  6 & 1 = 0        ← add 0
  dp[6] = dp[3] + 0 = 2  ✓

Pattern:
  Even numbers: same bit count as their half (last bit is 0)
  Odd numbers:  one more than their half  (last bit is 1)
```

## When To Use This Pattern

- When you need to compute a property for **all numbers 0..n**.
- When you spot a **recurrence relation** between a number
  and a smaller number (especially i//2 or i-1).
- When brute force would call a helper n times
  — DP amortizes the repeated work.
- When the problem says "O(n) without built-ins" — that's a
  hint to find the recurrence.

## The Approach

Create a dp array of length n+1, initialized to zero.
For each i from 1 to n, the answer is dp[i >> 1] + (i & 1).
Shifting right by 1 gives you i's "parent" (its value with
the last bit stripped), and (i & 1) adds back that last bit.
Return the completed dp array.

In [2]:
from typing import List  # standard collection types

In [3]:
def test_harness(func):
    tests = [
        # (input_n, expected, label)
        (0, [0],                    "edge: n=0"),
        (1, [0, 1],                 "n=1"),
        (2, [0, 1, 1],              "n=2"),
        (5, [0, 1, 1, 2, 1, 2],    "n=5, standard"),
        (8, [0,1,1,2,1,2,2,3,1],   "n=8, includes power-of-2"),
    ]
    passed = 0
    for n, expected, label in tests:
        result = func(n)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"  [{status}] {label}")
        if status == "FAILED":
            print(f"           got={result}")
            print(f"           exp={expected}")
    print(f"\n  {passed}/{len(tests)} tests passed")

In [7]:
def countBits(n: int) -> List[int]:
    """
    Return array where ans[i] = number of 1-bits in i,
    for all i in range [0, n].

    Approach: DP with bit-shift recurrence.
      dp[i] = dp[i >> 1] + (i & 1)
      i >> 1 is i//2; its bit count is already known.
      (i & 1) adds 1 if i is odd, 0 if even.

    Time:  O(n)
    Space: O(n) for the output array
    """
    ans = [0]* (n+1)
    offset = 1
    for i in range(1, n+1):
        if offset * 2 == i:
            offset = i
        ans[i] = 1 + ans[i - offset]
    return ans
r'''
[0]
[0, 1, 1]
[0, 1, 1, 2, 1, 2]
[0, 1, 1, 2, 1, 2, 2, 3, 1]
  [PASSED] edge: n=0
  [PASSED] n=1
  [PASSED] n=2
  [PASSED] n=5, standard
  [PASSED] n=8, includes power-of-2

  5/5 tests passed
'''


# Debug prints — expected values shown in comments
print(countBits(0))  # expected: [0]
print(countBits(2))  # expected: [0, 1, 1]
print(countBits(5))  # expected: [0, 1, 1, 2, 1, 2]
print(countBits(8))  # expected: [0,1,1,2,1,2,2,3,1]
test_harness(countBits)

[0]
[0, 1, 1]
[0, 1, 1, 2, 1, 2]
[0, 1, 1, 2, 1, 2, 2, 3, 1]
  [PASSED] edge: n=0
  [PASSED] n=1
  [PASSED] n=2
  [PASSED] n=5, standard
  [PASSED] n=8, includes power-of-2

  5/5 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(countBits)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (hammingWeight per number) | O(n * 32) | O(n) |
| DP with bit-shift recurrence | O(n) | O(n) |

## Real World Connection

In Citi's ETL pipelines, feature flags for each of 6,000 endpoints
are packed into integers. Building a bit-count lookup table once
and reusing it is far faster than recounting on every record scan.
On AWS, when Lambda logs are streaming in, pre-computed bitmask
popcount tables let analytics jobs classify permission bitmasks
for thousands of events per second without recalculating.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra